# Lezione 12 — Visione e linguaggio: progetto finale e chiusura del corso

## Obiettivi di apprendimento

- Capire il transfer learning con VGG16 (riutilizzare un modello preaddestrato su ImageNet per un problema nuovo con pochi dati).
- Capire come BLIP genera descrizioni di immagini in linguaggio naturale.
- Eseguire BLIP per davvero, gratuitamente, su un'immagine reale scaricata da URL.
- Ripassare, nell'ultima cella, tutto ciò che hai costruito lungo le 12 lezioni del corso.

## Cosa costruiremo

L'architettura concettuale di un classificatore di danni su aeromobili con VGG16 (transfer learning), e un generatore di descrizioni di immagini reale ed eseguibile con BLIP.

## Ripasso teorico: transfer learning con VGG16

Il caso di business reale di questo progetto (materiale B.8 del corso, `Aircraft Damage Dataset v1`) è automatizzare l'ispezione dei danni sugli aerei: classificare foto come ammaccatura (*dent*) o crepa (*crack*).

Invece di addestrare una CNN da zero (che richiederebbe moltissimi dati), riutilizziamo **VGG16**, una rete convoluzionale già addestrata su ImageNet (più di un milione di immagini, mille categorie). La strategia:

1. Caricare VGG16 senza il suo classificatore finale: `VGG16(weights='imagenet', include_top=False)`.
2. Congelare i suoi layer (`layer.trainable = False`) per non distruggere ciò che ha già imparato.
3. Aggiungere un classificatore proprio sopra: `Flatten()` → `Dense(512, relu)` → `Dropout(0.3)` → `Dense(512, relu)` → `Dropout(0.3)` → `Dense(1, sigmoid)`.
4. Addestrare **solo** questi nuovi layer con le foto di dent/crack.

Poiché il dataset completo di aerei è pesante per una sessione di lezione, qui vedrai l'architettura funzionare per davvero (VGG16 si scarica da solo, senza API key, viene incluso con Keras/TensorFlow), senza scaricare il dataset completo di foto di aerei.

In [ ]:
# VGG16 viene incluso con TensorFlow/Keras: non serve nessuna chiave API,
# vengono solo scaricati i pesi preaddestrati su ImageNet la prima volta che si usa.
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers

# Carichiamo VGG16 senza il suo classificatore finale, con la dimensione di immagine standard (224x224).
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Congeliamo tutti i layer del modello base: non vogliamo che l'addestramento
# successivo distrugga la conoscenza che VGG16 ha già appreso da ImageNet.
for layer in base_model.layers:
    layer.trainable = False

# Aggiungiamo il nostro classificatore sopra il modello congelato
# (per il caso reale: classificazione binaria dent vs crack).
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    loss="binary_crossentropy",
    optimizer=optimizers.Adam(learning_rate=0.0001),
    metrics=["accuracy"],
)

model.summary()

Osserva il riepilogo (`model.summary()`): la parte `vgg16` ha milioni di parametri, ma sono tutti **congelati** (non addestrabili) — solo i layer `Dense` che hai aggiunto alla fine sarebbero quelli addestrati con le tue immagini di dent/crack. Questo è il transfer learning: riutilizzi il "saper vedere" generico di VGG16 e addestri solo la parte specifica del tuo problema.

In un progetto reale, il passo successivo sarebbe:

```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    "aircraft_damage_dataset/train",   # cartelle train/dent e train/crack
    target_size=(224, 224),
    class_mode="binary",
)

model.fit(train_generator, validation_data=valid_generator, epochs=10)
```

Non lo eseguiamo qui perché richiede di scaricare il dataset completo di immagini di aerei — ma l'architettura sopra è esattamente quella che useresti.

## Ripasso teorico: BLIP, un modello che descrive immagini

**BLIP** (*Bootstrapping Language-Image Pre-training*) è un modello multimodale: riceve un'immagine e genera una descrizione in linguaggio naturale. Nel progetto originale viene usato due volte con due prompt distinti:

- `"This is a picture of"` → descrizione breve.
- `"This is a detailed photo showing"` → descrizione più dettagliata.

A differenza di VGG16 (che classifica tra categorie fisse), BLIP genera testo libero — risponde a "cosa c'è in questa immagine?", non a "a quale categoria appartiene?". Lo eseguiremo per davvero, su un'immagine pubblica scaricata da URL.

In [ ]:
# Installiamo il necessario (gratuito, senza API key).
!pip install -q transformers pillow requests

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from transformers import BlipProcessor, BlipForConditionalGeneration

# Carichiamo BLIP: processore (prepara immagine+testo) e modello (genera la descrizione).
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Scarichiamo un'immagine pubblica di esempio (foto di un cane su una spiaggia,
# un'immagine campione classica della documentazione di Hugging Face/BLIP).
url = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
respuesta = requests.get(url)
imagen = Image.open(BytesIO(respuesta.content)).convert("RGB")

imagen

In [ ]:
# Descrizione breve: il prompt corto spinge il modello a completare la frase.
inputs = processor(imagen, "a picture of", return_tensors="pt")
salida = blip_model.generate(**inputs, max_new_tokens=30)
caption = processor.decode(salida[0], skip_special_tokens=True)
print("Descrizione breve:", caption)

# Descrizione dettagliata: stesso modello, prompt di partenza diverso.
inputs = processor(imagen, "a detailed photo showing", return_tensors="pt")
salida = blip_model.generate(**inputs, max_new_tokens=60)
summary = processor.decode(salida[0], skip_special_tokens=True)
print("Descrizione dettagliata:", summary)

Confronta le due descrizioni: stesso modello, stesso peso, e l'unico cambiamento è il testo di partenza che hai dato al generatore. È la stessa idea che hai visto con il prompt di traduzione nella lezione precedente — il prompt determina il comportamento del modello — ma qui in un contesto in cui funziona SÌ come previsto, perché entrambi i prompt chiedono la stessa cosa (descrivere l'immagine) con un diverso livello di dettaglio, invece di chiedere compiti completamente diversi (tradurre vs. rispondere qualsiasi cosa).

## Chiusura: tutto ciò che hai costruito in questo corso

Lungo le 12 lezioni hai costruito, passo dopo passo:

1. **Forward propagation e funzioni di attivazione** (lezione 1) — come una rete neurale calcola una previsione, a mano con NumPy.
2. **Backpropagation** (lezione 2) — come una rete neurale impara, aggiustando i pesi con il problema XOR.
3. **Regressione e classificazione con Keras** (lezione 3) — da NumPy puro a un framework di alto livello, con dati reali (resistenza del calcestruzzo, cifre MNIST).
4. **Reti convoluzionali, CNN** (lezione 4) — estrazione di pattern visivi con `Conv2D` e `MaxPooling2D`.
5. **Self-attention** (lezione 5) — l'idea di Query/Key/Value dietro i moderni modelli di attenzione.
6. **LLM e chatbot con Hugging Face** (lezione 6) — il tuo primo chatbot da terminale, con Blenderbot e SmolLM2.
7. **Chatbot web con Flask** (lezione 7) — lo stesso chatbot, ora accessibile da un browser.
8. **Whisper e riconoscimento vocale** (lezione 8) — trascrizione automatica di audio, e riassunto di riunioni.
9. **Assistente vocale conversazionale** (lezione 9) — il ciclo completo voce → LLM → voce.
10. **RAG sui tuoi documenti** (lezione 10) — un chatbot che risponde basandosi sui tuoi PDF.
11. **Traduzione con Babel Fish** (lezione 11) — la stessa architettura vocale, applicata alla traduzione, e una lezione reale sulla verifica dei tuoi prompt.
12. **Visione e linguaggio: classificazione e captioning** (lezione 12, questa lezione) — transfer learning con VGG16 e descrizioni di immagini con BLIP.

Rivedi il capitolo di chiusura del manuale, [17_casos_de_uso_reales.md](../manual/17_casos_de_uso_reales_IT.md), per un ripasso dei 7 casi d'uso reali che hai costruito e per vedere come continuare a imparare.

**Continua con:**
- Capitolo del manuale: [16_vision_lenguaje_proyecto_final.md](../manual/16_vision_lenguaje_proyecto_final_IT.md)
- Notebook di pratica (capstone): [clase12_practica_vision_lenguaje_casos_uso.ipynb](../notebooks_practica/clase12_practica_vision_lenguaje_casos_uso_IT.ipynb)